# Predição de ativos da bolsa de valores

# Importa as bibliotecas

In [1]:
import sys
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
from plotly.offline import plot
import plotly.graph_objects as go

import datetime
#from pmdarima.arima import auto_arima # biblioteca que importa o arima
#import pmdarima.arima as pm # biblioteca que importa o arima
#import statsmodels
#from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error
import pandas as pd

from pandas_datareader import data 
import yfinance as yfin
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error


ModuleNotFoundError: No module named 'yfinance'

In [2]:
pip install yfinance

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 KB 1.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.0/508.0 KB 4.0 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.2/948.2 KB 13.1 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.9/147.9 KB 54.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 KB 38.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.2/115.2 KB 54.9 MB/s eta 0:00:00
  Using cached html5lib-1.1-py2.py3-none-any.whl (112 kB)
  Created wheel for peewee: filename=peewee-3.17.8-py3-none-any.whl size=138967 sha256=0ce73957b80a8727cfccbd4a1ddd925e6b8595b935cdd08559e07f7a5f5719a0
  Stored in directory: /root/.cache/pip/wheels/9f/cb/64/f38f9e4b9ef397c781cbd58e530f31841cb0a3740b5109bbb8
Successfully built peewee
  Attemp

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
import xgboost as xgb
import tensorflow as tf

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, Dense, Conv1D, LSTM, MaxPooling1D, Activation, MaxPooling1D, Dropout, Flatten, BatchNormalization, GRU, AveragePooling1D, SpatialDropout1D, GlobalAveragePooling1D
#from tensorflow.keras.initializers import he_uniform
from tensorflow import keras

## Análise e visualização de ativos

### Carrega Dados

In [ ]:
import yfinance as yf
import pandas as pd

yf.pdr_override()

cryptos=['BTC','ETH','SOL','ADA','TRX','FET','INJ']
peso_cryptos=np.ones(len(cryptos))*1/len(cryptos)

# adc o .SA no nome de cada ação para carregar no banco de dados
for i in range(np.size(cryptos)):
    cryptos[i]=cryptos[i]+"-USD"

cryptos_df = pd.DataFrame() 
for acao in cryptos:
     # Utilize o parâmetro 'interval' para especificar o intervalo de tempo
     cryptos_df[acao] = yf.download(acao, start='2017-01-01', interval='1d')['Close']

# substitui o .SA do nome de cada ação para visualização dos dados
for i in range(np.size(cryptos)):
    cryptos_df = cryptos_df.rename(columns={cryptos[i]:cryptos[i].replace('-USD', '')})

# Verifica como está o shape do dataframe
cryptos_df.shape

In [ ]:
cryptos_df

In [ ]:
sns.heatmap(cryptos_df.isnull())

In [ ]:
#apaga registros nulos
cryptos_df.dropna(inplace=True)
cryptos_df.to_csv('acoes.csv')
cryptos_df

### Visualização dos dados 

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(10,5))

columns=cryptos_df.columns
corrmat = cryptos_df[columns].corr()
mask= np.zeros_like(corrmat)
mask[np.triu_indices_from(mask)] = True

sns.heatmap(corrmat,
            vmax=1, vmin=-1,
            annot=True, annot_kws={'fontsize':15},
            cmap=sns.diverging_palette(20,220,as_cmap=True))
plt.show()

In [ ]:
cryptos_df.iloc[0].values

In [ ]:
# Normaliza os dados
cryptos_df_normalized = cryptos_df / cryptos_df.iloc[0].values

In [ ]:
cryptos_df_normalized

In [ ]:
#%% Carregando dados
cryptos_df=pd.read_csv('acoes.csv')
cryptos_df['Date']=pd.to_datetime(cryptos_df['Date'])
cryptos_df=cryptos_df.set_index('Date')
for col in cryptos_df.columns:
    cryptos_df[col]=cryptos_df[col].astype('float32')

In [ ]:
cryptos_df_normalized = cryptos_df / cryptos_df.iloc[0].values

In [ ]:
# Cria um gráfico de linha usando o Plotly
fig = go.Figure()
for cryptos in cryptos_df.columns:
#fig.add_trace(go.Scatter(x=cryptos_df.index, y=cryptos_df['IBOV'], mode='lines', name='Preço do Ibovespa'))
#fig.add_trace(go.Scatter(x=cryptos_df.index, y=cryptos_df['B3SA3'], mode='lines', name='Preço do B3SA3'))
    fig.add_trace(go.Scatter(x=cryptos_df_normalized.index, y=cryptos_df_normalized[cryptos], mode='lines', name=f'Preço do {cryptos}'))

# Configura o layout do gráfico
fig.update_layout(
    title='Preço do Ibovespa e ações',
    xaxis_title='Data',
    yaxis_title='Preço',
    showlegend=True
)

# Exibe o gráfico
fig.show()

## Indicadores de tendência e Séries Temporais 

Os swing traders geralmente utilizam médias móveis exponenciais (EMAs) de diferentes períodos para identificar tendências e sinais de compra e venda. As EMAs mais utilizadas pelos swing traders incluem:

-EMA de 9 períodos: A EMA de 9 períodos é comumente usada para identificar sinais de curto prazo e capturar movimentos rápidos do mercado. Pode fornecer sinais mais sensíveis e frequentes.

-EMA de 20 períodos: A EMA de 20 períodos é amplamente utilizada e considerada uma média móvel de curto prazo. É usada para identificar a direção da tendência de curto prazo e possíveis pontos de reversão.

-EMA de 50 períodos: A EMA de 50 períodos é frequentemente usada para identificar a direção da tendência de médio prazo. É uma média móvel amplamente observada pelos swing traders.

-EMA de 100 períodos: A EMA de 100 períodos é usada para identificar a direção da tendência de médio a longo prazo. É útil para identificar pontos de entrada e saída em operações de swing trading mais prolongadas.

-EMA de 200 períodos: A EMA de 200 períodos é uma das médias móveis mais amplamente observadas e é usada para identificar a direção da tendência de longo prazo. É frequentemente usada como um indicador-chave para determinar a tendência geral do mercado.

In [ ]:
import pandas as pd
import pandas_datareader as pdr
from pandas_datareader import data 
import datetime
import plotly.graph_objects as go

# Define o código de ticker da crypto
ticker = "BTC"


ticker=ticker+"-USD"

# Obtém os dados históricos da crypto
df = data.DataReader(ticker, start='2013-01-01')


In [ ]:
df.iloc[-10:,:]

In [ ]:
scaler = MinMaxScaler()
y=scaler.fit_transform(df['Close'].values.reshape(-1,1))

# Function to compute Simple Moving Average (SMA)b
def calculate_sma(data, window):
    return data['Close'].rolling(window=window).mean()

# Function to compute Relative Strength Index (RSI)
def calculate_rsi(data, window):
    delta = data['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

# Function to compute Moving Average Convergence Divergence (MACD)
def calculate_macd(data, short_window, long_window, signal_window):
    short_ema = data['Close'].ewm(span=short_window, min_periods=1, adjust=False).mean()
    long_ema = data['Close'].ewm(span=long_window, min_periods=1, adjust=False).mean()
    macd_line = short_ema - long_ema
    signal_line = macd_line.ewm(span=signal_window, min_periods=1, adjust=False).mean()
    return macd_line, signal_line

def calculate_bollinger_bands(data, window=20, num_std=2):
    sma = data['Close'].rolling(window=window).mean()
    std = data['Close'].rolling(window=window).std()
    upper_band = sma + num_std * std
    lower_band = sma - num_std * std
    return upper_band, lower_band

df['upper_band_bollinger'],df['lower_band_bollinger']=calculate_bollinger_bands(df)
# Calcula as EMAs de 9 e 20 períodos
df['EMA9'] = df['Close'].ewm(span=9).mean()
df['EMA20'] = df['Close'].ewm(span=20).mean()
df['EMA50'] = df['Close'].ewm(span=50).mean()
df['EMA100'] = df['Close'].ewm(span=100).mean()
df['EMA200'] = df['Close'].ewm(span=200).mean()
df['MA111'] = df['Close'].rolling(window=111).mean()
df['MA350'] = df['Close'].rolling(window=350).mean() 
df['MA111*2'] = df['Close'].rolling(window=111).mean() * 2
df['MA350*2'] = df['Close'].rolling(window=350).mean() * 2


# Calculate RSI (14-period)
df['RSI_14'] = calculate_rsi(df, window=14)

# Calculate MACD (12, 26, 9)
df['MACD'], df['Signal'] = calculate_macd(df, short_window=12, long_window=26, signal_window=9)


In [ ]:
# Cria um gráfico de candlestick usando o Plotly
fig = go.Figure(data=[go.Candlestick(x=df.index,
                open=df['Open'],
                high=df['High'],
                low=df['Low'],
                close=df['Close'],
                name='Candlestick')])

# Adiciona as médias móveis ao gráfico
fig.add_trace(go.Scatter(x=df.index, y=df['EMA9'], mode='lines', name='EMA9'))
fig.add_trace(go.Scatter(x=df.index, y=df['EMA20'], mode='lines', name='EMA20'))
fig.add_trace(go.Scatter(x=df.index, y=df['EMA50'], mode='lines', name='EMA50'))
fig.add_trace(go.Scatter(x=df.index, y=df['EMA100'], mode='lines', name='EMA100'))
fig.add_trace(go.Scatter(x=df.index, y=df['EMA200'], mode='lines', name='EMA200'))
fig.add_trace(go.Scatter(x=df.index, y=df['MA111'], mode='lines', name='MA111'))
fig.add_trace(go.Scatter(x=df.index, y=df['MA350'], mode='lines', name='MA350'))
fig.add_trace(go.Scatter(x=df.index, y=df['MA111*2'], mode='lines', name='MA111*2'))
fig.add_trace(go.Scatter(x=df.index, y=df['MA350*2'], mode='lines', name='MA350*2'))

fig.add_trace(go.Scatter(x=df.index, y=df['upper_band_bollinger'], mode='lines', name='upper band bollinger'))
fig.add_trace(go.Scatter(x=df.index, y=df['lower_band_bollinger'], mode='lines', name='lower band bollinger'))

if ticker=="^BVSP":
    ticker='IBOV'

# Configura o layout do gráfico
fig.update_layout(
    width=1500,
    height=700,
    title=f"Gráfico de Candlestick e Médias Móveis para {ticker}",
    xaxis_title="Data",
    yaxis_title="Preço",
    showlegend=True
)

# Exibe o gráfico
fig.update_layout(xaxis_rangeslider_visible=False)
fig.show()


In [ ]:
# Cria um gráfico de candlestick usando o Plotly
fig = go.Figure(data=[go.Candlestick(x=df.index,
                open=df['Open'],
                high=df['High'],
                low=df['Low'],
                close=df['Close'],
                name='Candlestick')])

# Adiciona as médias móveis ao gráfico
fig.add_trace(go.Scatter(x=df.index, y=df['upper_band_bollinger'], mode='lines', name='upper band bollinger'))
fig.add_trace(go.Scatter(x=df.index, y=df['lower_band_bollinger'], mode='lines', name='lower band bollinger'))


# Configura o layout do gráfico
fig.update_layout(
    width=1500,
    height=700,
    title=f"Gráfico de Candlestick e Médias Móveis para {ticker}",
    xaxis_title="Data",
    yaxis_title="Preço",
    showlegend=True
)
fig.update_layout(xaxis_rangeslider_visible=False)

# Exibe o gráfico
fig.show()


In [ ]:
def backtest(close_price_list, labels, initial_capital=10000, verbose=0):
    """
    Backtests a trading strategy based on buy/sell signals.

    Args:
        close_price_list: A list of closing prices for each period.
        labels: A list of trading signals ("BUY", "SELL", or "HOLD").
        initial_capital: The initial capital for the backtest.
        verbose: An integer controlling verbosity: 0 (no output), 1 (prints buy/sell actions), 2 (prints all actions and portfolio value).

    Returns:
        A list of portfolio values at each period.
    """
    cash = initial_capital  # Dinheiro em caixa
    shares = 0  # Quantidade de ações
    portfolio_values = []  # Histórico do valor total do portfólio
    
    for i, label in enumerate(labels):
        price = close_price_list[i]
        
        if label == "BUY" and cash >= price:
            shares_to_buy = cash // price
            shares += shares_to_buy
            cash -= shares_to_buy * price
            if verbose >= 1:
                print(f"BUY: Comprado {shares_to_buy} ações a R$ {price:.2f}")

        elif label == "SELL" and shares > 0:
            cash += shares * price
            shares = 0
            if verbose >= 1:
                print(f"SELL: Vendido {shares} ações a R$ {price:.2f}")

        # Valor atual do portfólio
        portfolio_value = cash + shares * price
        portfolio_values.append(portfolio_value)
        if verbose >= 2:
            print(f"Periodo {i+1}: Portfólio = R$ {portfolio_value:.2f}")

    # Resultado final
    final_value = cash + shares * close_price_list[-1]
    if verbose >= 1:
        print(f"Valor final do portfólio: R$ {final_value:.2f}")
        print(f"Retorno total: {((final_value - initial_capital) / initial_capital) * 100:.2f}%")

    return portfolio_values


In [ ]:
import os, sys
processing_source_path = os.path.abspath('Processing/')
if(processing_source_path not in sys.path):
    sys.path.append(processing_source_path)
from DataLoaderPipeline import FeaturesDataGenerator
features_dg=FeaturesDataGenerator()
# Dados de exemplo
close_price_list = df['Close'].values

#labels = labelling_method(close_price_list, 60)
label_data = features_dg.label_data(close_prices=df['Close'], window= 16)

labels=[("BUY" if np.allclose(label, [0, 1, 0]) else "SELL" if np.allclose(label, [0, 0, 1]) else "HOLD") 
        for label in label_data]# Realizar o backtest

print(np.unique(labels, return_counts=True))
# Realizar o backtest
portfolio_values = backtest(close_price_list, labels, initial_capital=100)

# Gráfico do desempenho
import plotly.graph_objects as go

#fig = go.Figure()

fig = go.Figure(data=[go.Candlestick(x=df.index,
                open=df['Open'],
                high=df['High'],
                low=df['Low'],
                close=df['Close'],
                name='Candlestick')])

# Valor do portfólio
#fig.add_trace(go.Scatter(
#    x=df.index,
#    y=portfolio_values,
#    mode='lines',
#    name='Valor do Portfólio'
#))

# Configuração do layout
fig.update_layout(
    title="Backtest de Estratégia de BUY, SELL, HOLD",
    xaxis_title="Data",
    yaxis_title="Valor",
    showlegend=False,
    width=1700,
    height=600,
    margin=dict(l=50, r=50, t=50, b=50)  # Margem esquerda, direita, superior e inferior
)


# Adiciona os marcadores BUY, SELL e HOLD
for i, label in enumerate(labels):
    if label == "BUY":
        fig.add_trace(go.Scatter(
            x=[df.index[i]],
            y=[df['Close'].iloc[i]- 0.03*df['Close'].iloc[i]] ,
            mode='markers',
            marker=dict(color='green', size=10, symbol='triangle-up'),
            name='BUY'
        ))
    elif label == "SELL":
        fig.add_trace(go.Scatter(
            x=[df.index[i]],
            y=[df['Close'].iloc[i]+ 0.03*df['Close'].iloc[i]],
            mode='markers',
            marker=dict(color='red', size=10, symbol='triangle-down'),
            name='SELL'
        ))
fig.update_layout(xaxis_rangeslider_visible=False)
fig.show()

In [ ]:

# Calcular a variação percentual entre o preço de fechamento de um dia e o dia seguinte
df['Variation'] = ((df['Close'] - df['Close'].shift(+1)) / df['Close'])*100
#pred_data_df['Variacao'] = ((pred_data_df['pred']-pred_data_df['pred'].shift(+1)) / pred_data_df['pred'])*100


# Thresholds for variation classification
positive_threshold = 0.03
negative_threshold = -0.03

# Create a new column for classified variation
df['Classification'] = df['Variation'].apply(lambda x: 
                                            1 if x > positive_threshold else (
                                                -1 if x < negative_threshold else 0))

#df['week']=df.index.week
df['weekday']=df.index.weekday+1
df['day']=df.index.day
df['month']=df.index.month
df['quarter']=df.index.quarter

df=df.reset_index().fillna(0)
df


## Data preprocessing to stock forecast

In [ ]:
del acao, axes, close_price_list, col, columns, corrmat, cryptos, cryptos_df, cryptos_df_normalized, fig, features_dg

In [ ]:
import requests
import pandas as pd
import numpy as np

# Lista de criptomoedas
ticker='FET'
cryptos = [ticker]
ticker=ticker+"-USDT"

peso_cryptos = np.ones(len(cryptos)) * 1 / len(cryptos)

# Adiciona o par USD para cada cripto
cryptos = [crypto + "USDT" for crypto in cryptos]

# Função para baixar dados históricos da Binance API
def get_binance_data(symbol, interval='1d', start_time='2017-01-01'):
    base_url = "https://api.binance.com/api/v3/klines"
    end_time = int(pd.Timestamp.now().timestamp() * 1000)  # Data atual

    # Converte a data de início para timestamp
    start_timestamp = int(pd.Timestamp(start_time).timestamp() * 1000)

    # Lista para armazenar os dados
    data_list = []

    # Faça requisições iterativas até obter todos os dados
    while start_timestamp < end_time:
        params = {
            'symbol': symbol,
            'interval': interval,
            'startTime': start_timestamp,
            'endTime': start_timestamp + (1000 * 60 * 60 * 24 * 365),  # Intervalo de 1 ano
            'limit': 1000  # Máximo de registros por chamada
        }

        # Coleta dados da API
        response = requests.get(base_url, params=params)
        data = response.json()

        # Verifica se a resposta da API está vazia
        if not data:
            break

        # Converte os dados em DataFrame
        df = pd.DataFrame(data, columns=[
            'timestamp', 'open', 'high', 'low', 'close', 'volume',
            'close_time', 'quote_asset_volume', 'number_of_trades',
            'taker_buy_base', 'taker_buy_quote', 'ignore'
        ])

        # Verifica se o DataFrame está vazio
        if df.empty:
            break

        # Adiciona os dados à lista
        data_list.append(df)

        # Atualiza o start_timestamp para a próxima requisição
        if not df.empty:
            start_timestamp = int(df['timestamp'].iloc[-1]) + 1000  # Adiciona 1 segundo ao último timestamp
        else:
            break

    # Concatena todos os DataFrames
    if data_list:
        df = pd.concat(data_list, ignore_index=True)
    else:
        df = pd.DataFrame()

    # Formata e filtra os dados necessários
    if not df.empty:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
        df.set_index('timestamp', inplace=True)
        df['close'] = df['close'].astype(float)
        df['low'] = df['low'].astype(float)
        df['high'] = df['high'].astype(float)
        df['open'] = df['open'].astype(float)
        df['volume'] = df['volume'].astype(float)
    return df


# Intervalos disponíveis
intervals = ['1m', '3m', '5m', '15m', '30m', '1h', '2h', '4h', '6h', '8h', '12h', '1d', '3d', '1w', '1M']

#choice = int(input("Escolha um intervalo (número): "))
#interval = intervals[choice - 1]
interval ='4h'

# Baixar e consolidar dados em um DataFrame
cryptos_df = pd.DataFrame()
for crypto in cryptos:
    data = get_binance_data(crypto, start_time='2019-01-01', interval=interval)
    # Ajusta o nome da coluna removendo "USDT" antes de adicionar ao DF
    #data.columns = [crypto.replace('USDT', '')]
    if cryptos_df.empty:
        cryptos_df = data
    else:
        cryptos_df = pd.concat([cryptos_df, data], axis=1)

if not cryptos_df.empty:
    cryptos_df = cryptos_df.rename(columns={
        'open': 'Open',
        'high': 'High',
        'low': 'Low',
        'close': 'Close',
        'volume': 'Volume',
    })
    # Visualização dos dados
    cryptos_df=cryptos_df.rename_axis('Date')
    print(cryptos_df.shape)
else:
    print("Nenhum dado encontrado para o símbolo e intervalo especificados.")

df=cryptos_df
df.reset_index(inplace= True)


### Pré-processamento dos dados para predição 

In [ ]:
import os, sys
processing_source_path = os.path.abspath('Processing/')
if(processing_source_path not in sys.path):
    sys.path.append(processing_source_path)
from DataLoaderPipeline import FeaturesDataGenerator, scrapingHistoricalData

In [ ]:
SHD=scrapingHistoricalData()
df=SHD.get_crypto_historical_data(['BTC'], interval, '2015-01-01')

In [ ]:
split_data=FeaturesDataGenerator().split_data

In [ ]:
# separa os dados em Treino e Teste
split_data=FeaturesDataGenerator().split_data
X_train, X_val, T_train, T_val=split_data(df['Close'].values, df['Date'], factor=0.7)
X_val, X_test, T_val, T_test=split_data(X_val.T, T_val, factor=0.7)

print('Train data shape',X_train.shape)
print('Vall data shape',X_val.shape)
print('Test data shape',X_test.shape)


In [ ]:
#features_indicators=FeaturesDataGenerator().features_name
#features_indicators=['Data_lookback','variations','EMA9', 'EMA20', 'EMA50', 'EMA100', 'EMA200']
features_indicators=['Data_lookback', 'Volume',
                     'EMA9', 'EMA20', 'EMA50', 'EMA100', 'EMA200','MA111', 'MA350','MACD']

#features_indicators=['Data_lookback','EMA9', 'EMA20', 'EMA50', 'EMA100', 'EMA200','MA111', 'MA350','MACD',]
features_indicators=['Data_lookback', 'Volume', 'Low', 'High', 'Open','EMA9', 'EMA20', 'EMA50', 'EMA100', 'EMA200','MA111', 'MA350','MACD']




features_indicators=['Data_lookback', 'Open', 'High', 'Low', 'Volume', 
                    'EMA9', 'EMA20', 'EMA50', 'EMA100', 'EMA200', 'MA111', 'MA350', 
                    'RSI_14',  'Chaikin_Money_Flow',
                    'MACD', 'MACD_Signal', 'MACD_Histogram', 
                    'Stochastic_K', 'Stochastic_D', 
                    'Bollinger_Bands_Upper', 'Bollinger_Bands_Middle', 'Bollinger_Bands_Lower',
                    'CCI','Williams_R',  'ROC',  'PPO', 'Chaikin_Money_Flow']

features_indicators=['Data_lookback', 'Open', 'High', 'Low', 'Volume']

features_indicators=[
                    'EMA9', 'EMA20', 'EMA50', 'EMA100', 'EMA200',
                    'MA111', 'MA350', 
                    #'RSI_14',  
                    #'MACD', 'MACD_Signal', 'MACD_Histogram',
                    'Bollinger_Bands_Upper', 'Bollinger_Bands_Middle', 'Bollinger_Bands_Lower']
features_indicators=[ 'CCI','Williams_R',  'ROC',  'PPO', 'Chaikin_Money_Flow']


features_indicators=['Data_lookback', 'Open', 'High', 'Low', 
                    'EMA9', 'EMA20', 'EMA50', 'EMA100', 'EMA200', 'MA111', 'MA350', 
                    'RSI_14',  'Chaikin_Money_Flow',
                    'MACD', 'MACD_Signal', 'MACD_Histogram', 
                    'Stochastic_K', 'Stochastic_D', 
                    'Bollinger_Bands_Upper', 'Bollinger_Bands_Middle', 'Bollinger_Bands_Lower',
                    'CCI',  'ROC',  'PPO', 'Chaikin_Money_Flow']#, 'Williams_R']

pred_days = int(24/4 * 2.5) 
buy_sell_threshold=[0.05,-0.05]
lookback = 24
batch_size = 16
shuffle = True

min_norm=-1
max_norm=1

X_data_gen_train = FeaturesDataGenerator(df[df['Date'].isin(T_train)].iloc[:,:], lookback = lookback, pred_days = pred_days, buy_sell_threshold=buy_sell_threshold, shuffle= shuffle, batch_size=batch_size, selected_features = features_indicators, data_augmentation=False, min_max_norm=[min_norm, max_norm])
X_data_gen_val = FeaturesDataGenerator(df[df['Date'].isin(T_val)].iloc[:,:], lookback = lookback, pred_days = pred_days,  buy_sell_threshold=buy_sell_threshold, shuffle= shuffle, batch_size=batch_size, selected_features = features_indicators, data_augmentation=False, min_max_norm=[min_norm, max_norm])
X_data_gen_test = FeaturesDataGenerator(df[df['Date'].isin(T_test)].iloc[:,:], lookback = lookback, pred_days = pred_days,  buy_sell_threshold=buy_sell_threshold, shuffle= False, batch_size=batch_size, selected_features = features_indicators, data_augmentation=False, min_max_norm=[min_norm, max_norm])

In [ ]:
plt.plot(X_data_gen_train.features[:,-1, 0:])
plt.show()

In [ ]:
# Dados de exemplo
data_idx = -500
input_data = X_data_gen_test.InputData['Close'].iloc[data_idx:].values
date = X_data_gen_test.InputData['Date'].iloc[data_idx:]
#labels = labelling_method(close_price_list, 60)

labels=[("BUY" if np.allclose(label, [0, 1, 0]) else "SELL" if np.allclose(label, [0, 0, 1]) else "HOLD") 
        for label in X_data_gen_test.y_classification]# Realizar o backtest


labels=labels[data_idx:]
print(np.unique(labels, return_counts=True))
#portfolio_values = backtest(input_data, labels, initial_capital=100000)

# Gráfico do desempenho
import plotly.graph_objects as go

#fig = go.Figure()

fig = go.Figure(data=[go.Candlestick(x=X_data_gen_test.InputData['Date'].iloc[data_idx:],
                open=X_data_gen_test.InputData['Open'].iloc[data_idx:],
                high=X_data_gen_test.InputData['High'].iloc[data_idx:],
                low=X_data_gen_test.InputData['Low'].iloc[data_idx:],
                close=X_data_gen_test.InputData['Close'].iloc[data_idx:],
                name='Candlestick')])

# Valor do portfólio
#fig.add_trace(go.Scatter(
#    x=X_data_gen_test.InputData['Date'],
#    y=portfolio_values,
#    mode='lines',
#    name='Valor do Portfólio'
#))

# Configuração do layout
fig.update_layout(
    title="Backtest de Estratégia de BUY, SELL, HOLD",
    xaxis_title="Data",
    yaxis_title="Valor",
    showlegend=False,
    width=1700,
    height=600,
    margin=dict(l=50, r=50, t=50, b=50)  # Margem esquerda, direita, superior e inferior
)


# Adiciona os marcadores BUY, SELL e HOLD
for i, label in enumerate(labels):
    if label == "BUY":
        fig.add_trace(go.Scatter(
            x=[date.iloc[i]],
            y=[input_data[i] - 0.05*input_data[i]],
            mode='markers',
            marker=dict(color='green', size=10, symbol='triangle-up'),
            name='BUY'
        ))
    elif label == "SELL":
        fig.add_trace(go.Scatter(
            x=[date.iloc[i]],
            y=[input_data[i] + 0.05*input_data[i]],
            mode='markers',
            marker=dict(color='yellow', size=10, symbol='triangle-down'),
            name='SELL'
        ))
fig.update_layout(xaxis_rangeslider_visible=False)
fig.show()


In [ ]:
X_data_gen_train.features[:,:,:].shape

In [ ]:
plt.plot(X_data_gen_train.features[:,-1, :])
plt.show()


In [ ]:
for idx, data_2d in enumerate(np.array(X_data_gen_train[15][0][:3])):
    plt.figure(figsize=(15,8))
    # Visualize the image
    plt.imshow(data_2d.T, cmap='gray', interpolation='nearest')
    plt.title('label: {}'.format(X_data_gen_train[15][1][idx]))

    #plt.colorbar()
    plt.show()

In [ ]:
input_data= X_data_gen_test.InputData['Close'][lookback:].values.reshape(-1,1)
input_data.shape


In [ ]:
np.diff(X_data_gen_test.InputData['Close'])[lookback-1:].shape

In [ ]:
#aux_data=np.hstack([X_data_gen_test.variations.reshape(-1,1),X_data_gen_test.features[:,0].reshape(-1,1),X_data_gen_test.y_classification])
input_data= X_data_gen_test.InputData['Close'][lookback:].values.reshape(-1,1)
variations = np.diff(X_data_gen_test.InputData['Close'])[lookback-1:]

aux_data=np.hstack([variations.reshape(-1,1), input_data, X_data_gen_test.y_classification])

#recomentations_2=pd.DataFrame(data=aux_data,columns=['var','X', 'Hold','Buy','Strong_Buy','Sell','Strong_Sell'])
recomentations_2=pd.DataFrame(data=aux_data,columns=['var','X', 'Hold','Buy','Sell'])
recomentations_2.tail(20)    


In [ ]:
X_data_gen_test.InputData

In [ ]:
X_data_gen_test.InputData

In [ ]:
#recomentations_2=pd.DataFrame(data=np.hstack([np.roll(X_data_gen_train.variations.reshape(-1,1),1)[lookback:],X_data_gen_train.features[:,lookback-1].reshape(-1,1),X_data_gen_train.y_classification]),columns=['var','X', 'Hold','Buy','Sell'])
#recomentations_2.head(10)

In [ ]:
'''variations = ((X_train - np.roll(X_train, 1)) / X_train) * 100
recomendations=pd.DataFrame(data=np.hstack([np.vstack([variations[lookback:-pred_days],X_train[lookback:-pred_days]]).T,X_data_gen_train.comput_outputs(X_train)]), columns=['var','X', 'Hold','Buy','Sell'])
recomendations.head(10)'''

###  Get imbalanced database 

In [ ]:
Y_train_categorical=np.argmax(X_data_gen_train.y_classification,axis=1)
classes, counts=np.unique(Y_train_categorical,return_counts=True) 

fig, ax = plt.subplots()
plt.bar(classes,counts)
ax.set_ylabel('number of classes')
ax.set_title('Traning Dataset classes')
plt.show()

In [ ]:
Y_val_categorical=np.argmax(X_data_gen_val.y_classification,axis=1)
classes, counts=np.unique(Y_val_categorical,return_counts=True) 

fig, ax = plt.subplots()
plt.bar(classes,counts)
ax.set_ylabel('number of classes')
ax.set_title('Validation Dataset classes')
plt.show()

In [ ]:
Y_test_categorical=np.argmax(X_data_gen_test.y_classification,axis=1)
classes, counts=np.unique(Y_test_categorical,return_counts=True) 

fig, ax = plt.subplots()
plt.bar(classes,counts)
ax.set_ylabel('number of classes')
ax.set_title('Validation Dataset classes')
plt.show()

In [ ]:
from sklearn.utils import class_weight
n_classes, counts=np.unique(Y_train_categorical,return_counts=True) 

output_class_weights = class_weight.compute_class_weight('balanced', classes=n_classes, y=np.argmax(X_data_gen_train.y_classification,axis=1))
print(output_class_weights)
weighted_categorical_crossentropy_loss= X_data_gen_train.weighted_categorical_crossentropy(output_class_weights)

## Neural Networkt aproach

In [ ]:
X_data,y_data=X_data_gen_train[0]
n_classes=y_data.shape[1]
n_classes

### Models 1D

#### CNN_LSTM

In [ ]:
# Model name
model_name = "CNN_LSTM_MultiHead"
#np.random.seed(42)

# Define CNN-LSTM feature extraction function
def CNN_LSTM(inputs):
  x = Conv1D(64, kernel_size=1, strides=1, activation='relu')(inputs)
  x = BatchNormalization()(x)
  x = MaxPooling1D(2)(x)
  x = Dropout(0.5)(x)
  return x

def CNN_bracnh(Features):
    x = Conv1D(64, kernel_size=1, strides=1, activation='relu')(Features)
    x = BatchNormalization()(x)
    x = MaxPooling1D(1)(x)
    x = Dropout(0.5)(x)
    x = Flatten()(x)
    return x
  
# Define a single function for both heads (regression and classification)
def head(features, n_outputs, activation, name= None):
  x = Dense(256)(features)
  x = BatchNormalization()(x)
  x = Activation('relu')(x)
  output = Dense(n_outputs)(x)

  output = Activation(activation, name=name)(output)  # Dynamic output name

  return output

# Create the full model
input_shape = (X_data.shape[1],X_data.shape[2])  # Assuming your input shape
input_data = Input(shape=input_shape)

Features = CNN_LSTM(input_data)
#features_branch1=CNN_bracnh(Features)
features_branch2=CNN_bracnh(Features)
# Create separate heads with appropriate number of outputs and activations
#regression_output = head(features_branch1, n_outputs=pred_days, activation='linear',name='regression_head')
classification_output = head(features_branch2, n_outputs=n_classes, activation='softmax',name='classification_head')

# Create the model with two heads
model_CNN_LSTM = Model(inputs=input_data, outputs=[classification_output])
model_CNN_LSTM._name = model_name

# Print model summary
model_CNN_LSTM.summary()

In [ ]:
X_data.shape

In [ ]:
# Model name
model_name = "CNN_MultiHead"
#np.random.seed(42)

def common_layers(input1):
        """Common layers to the network model

        Returns:
            Graph: the common layers model
        """

        ##################################################################
        # CNN architecture
        conv_layer = Conv1D(64, (3), padding="same")(input1)
        conv_layer = BatchNormalization()(conv_layer)
        conv_layer = Activation('relu')(conv_layer)
        conv_layer = MaxPooling1D(pool_size=(3), strides=1, padding="same")(conv_layer)
        conv_layer = Conv1D(128, (3), padding="same")(conv_layer)
        conv_layer = BatchNormalization()(conv_layer)
        conv_layer = Activation('relu')(conv_layer)
        conv_layer = MaxPooling1D(pool_size=(3), strides=2, padding="same")(conv_layer)
        
        ##################################################################

        return conv_layer

def head_layer(conv_layer, num_classes, activation='linear', output_name=None):
    
    head = Conv1D(64, (3), padding="same")(conv_layer)
    head = BatchNormalization()(head)
    head = Activation('relu')(head)
    head = MaxPooling1D(pool_size=(3), strides=2, padding="same")(head)

    head = Dropout(0.5)(head)
    head = Flatten()(head)

    x = Dense(256)(head)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Dropout(0.5)(x)

    x = Dense(128)(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Dropout(0.5)(x)

    x = Dense(num_classes)(x)
    x = Activation(activation, name=output_name)(x)

    return x

# Create the full model
input_data = Input(shape=input_shape)

Features = common_layers(input_data)
# Create separate heads with appropriate number of outputs and activations
regression_output = head_layer(Features, num_classes=pred_days, activation='linear',output_name='regression_head')
classification_output = head_layer(Features, num_classes=n_classes, activation='softmax',output_name='classification_head')

# Create the model with two heads
model_CNN_MultiHead = Model(inputs=input_data, outputs=[classification_output])
model_CNN_MultiHead._name = model_name

# Print model summary
model_CNN_MultiHead.summary()

#### MultiLayer Perceptron

In [ ]:
# Model name
model_name = "MLP"

# Input layer
input_shape = (X_data.shape[1],X_data.shape[2])  # Assuming your input shape
input_data = Input(shape=input_shape)

# Adicione uma camada de flatten
x = Flatten()(input_data)
# Dense layer 1 with batch normalization
x = Dense(64)(x)
x = BatchNormalization()(x)
activation1 = Activation('relu')(x)

#Dense layer 2 with batch normalization and dropout
x = Dense(32)(activation1)
x = BatchNormalization()(x)
activation2 = Activation('relu')(x)
x = Dropout(0.5)(activation2)


# Define a single function for both heads (regression and classification)
def head(features, n_outputs, activation='linear', name= None):
  """
  Defines a dense layer head for either regression or classification.

  Args:
      features: Input tensor from the feature extraction part.
      n_outputs: Number of outputs for the head.
      activation: Activation function for the output layer (defaults to 'linear' for regression).

  Returns:
      A Keras functional API model representing the head with its output.
  """
  
  x = Dense(32)(features)
  x = BatchNormalization()(x)
  x = Activation('relu')(x)
  
  output = Dense(n_outputs)(x)
  output = Activation(activation, name=name)(output)  # Dynamic output name

  return output


# Create separate heads with appropriate number of outputs and activations
#regression_output = head(activation2, n_outputs=pred_days, activation='linear',name='regression_head')
classification_output = head(activation2, n_outputs=n_classes, activation='softmax',name='classification_head')

# Create the model
model_MLP = Model(inputs=input_data, outputs=[classification_output])
model_MLP._name = model_name

# Print model summary
model_MLP.summary()


In [ ]:
model_name = "simple_MLP"
input_shape = (X_data.shape[1],X_data.shape[2])  # Assuming your input shape

model_simple_MLP = Sequential([
    Dense(64, activation='relu', input_shape=input_shape),
    Dense(32, activation='relu'),
    Dense(n_classes, activation='softmax')
])

model_simple_MLP._name = model_name
model_simple_MLP.summary()

#### Long Short-Term Memory (LSTM) 

In [ ]:
model_name= "LSTM"
# Camada de entrada
input_shape = (X_data.shape[1],X_data.shape[2])  # Assuming your input shape
input_data = Input(shape=input_shape)

# Camada LSTM
lstm_output  = LSTM(64, return_sequences=True)(input_data)
bn_lstm = BatchNormalization()(lstm_output)
activation_lstm = Activation('relu')(bn_lstm)
Dropout_output = Dropout(0.5)(activation_lstm)

lstm_output2  = LSTM(64, activation='relu', return_sequences=False)(Dropout_output)
Dropout_output = Dropout(0.5)(lstm_output2)

#lstm_output3  = LSTM(64, activation='relu', return_sequences=True)(Dropout_output)
#Dropout_output = Dropout(0.1)(lstm_output3)

#lstm_output  = LSTM(4, activation='tanh')(Dropout_output)
#Dropout_output = Dropout(0.5)(lstm_output)

dense_output = Dense(32, activation='relu')(Dropout_output)
Dropout_output=Dropout(0.5)(dense_output)

#dense_output = Dense(8, activation='relu')(Dropout_output)
#Dropout_output=Dropout(0.5)(dense_output)
# Camada de saída
output = Dense(n_classes, activation='softmax')(Dropout_output)

model_LSTM = Model(inputs=input_data, outputs=output)
model_LSTM._name = model_name
model_LSTM.summary()

#### Long Short-Term Memory (GRU) 

In [ ]:
model_name= "GRU"
# Camada de entrada
input_shape = (X_data.shape[1],X_data.shape[2])  # Assuming your input shape
input_data = Input(shape=input_shape)
# Camada LSTM
GRU_output  = GRU(64)(input_data)

bn_lstm = BatchNormalization()(GRU_output)
activation_lstm = Activation('relu')(bn_lstm)
Dropout_output = Dropout(0.5)(activation_lstm)

output = Dense(n_classes, activation='softmax')(Dropout_output)

model_GRU = Model(inputs=input_data, outputs=output)
model_GRU._name = model_name
model_GRU.summary()

#### LSTM with Attention layer

In [ ]:
@tf.keras.utils.register_keras_serializable(package="Custom", name="Attention_")
class Attention_(tf.keras.layers.Layer):
    def __init__(self, units,  **kwargs):
        super(Attention_, self).__init__( **kwargs)
        self.units = units
        self.W = tf.keras.layers.Dense(units)
        self.V = tf.keras.layers.Dense(1)

    def build(self, input_shape):
        self.W.build(input_shape)
        self.V.build(input_shape)

    def call(self, inputs):
        # Compute attention scores
        score = tf.nn.tanh(self.W(inputs))
        attention_weights = tf.nn.softmax(self.V(score), axis=-1)

        # Apply attention weights to input
        context_vector = attention_weights * inputs
        context_vector = tf.reduce_sum(context_vector, axis=-1)

        return context_vector

    @classmethod
    def from_config(cls, config):
        return cls(**config)


In [ ]:
X_data.shape

In [ ]:
model_name= "LSTM_AT"
# Camada de entrada
input_shape = (X_data.shape[1],X_data.shape[2])  # Assuming your input shape
input_data = Input(shape=input_shape)

# Dense layer 1 with batch normalization
x = Dense(64)(input_data)
x = BatchNormalization()(x)
x = Activation('relu')(x)

#Dense layer 2 with batch normalization and dropout
x = Dense(32)(x)
x = BatchNormalization()(x)
X = Activation('relu')(x)
x = Dropout(0.5)(x)
x = Dense(32)(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)

LSTM_output = tf.keras.layers.LSTM(64, return_sequences=True)(x)

bn_lstm = BatchNormalization()(LSTM_output)
activation_lstm = Activation('relu')(bn_lstm)
X = Dropout(0.5)(activation_lstm)

x = Attention_(64)(X)

x = tf.keras.layers.Dense(n_classes,activation='softmax')(x)

model_LSTM_AT = tf.keras.Model(inputs=input_data, outputs=x)
model_LSTM_AT._name = model_name

model_LSTM_AT.summary()

### Models 2D

In [ ]:
'''
def CNN_model():
    model = Sequential()
    model.add(Conv2D (32,5,5, padding='same', input_shape=(1,5, 4), activation='relu')) 
    model.add(MaxPooling2D(pool_size=(2,2), padding='same'))
    model.add(Dense (200, activation='relu'))
    model.add(Dropout (0.2))
    model.add(Flatten())
    model.add(Dense (200, activation=' relu'))
    model.add(Dense (3, activation='softmax'))
    model.compile(loss='categorical_crossentropy', optimizer='Adam', metrics=['accuracy'])
    return model
    '''

### Hyperparams and trainnig

**Metrics and Loss Functions**  

In [ ]:
from tensorflow.keras import backend as K

def matthews_correlation_coefficient(y_true, y_pred):
    tp = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    tn = K.sum(K.round(K.clip((1 - y_true) * (1 - y_pred), 0, 1)))
    fp = K.sum(K.round(K.clip((1 - y_true) * y_pred, 0, 1)))
    fn = K.sum(K.round(K.clip(y_true * (1 - y_pred), 0, 1)))

    num = tp * tn - fp * fn
    den = (tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)
    return num / K.sqrt(den + K.epsilon())

def R2(y_true, y_pred): # squareds Pearson's correlation coef 
    SS_res =  K.sum(K.square( y_true-y_pred ))
    SS_tot = K.sum(K.square( y_true - K.mean(y_true) ) )
    return ( 1 - SS_res/(SS_tot - K.epsilon()) )

def NRMSE(y_true, y_pred): # normalized_root_mean_squared_error
    return 1-K.sqrt(K.mean(K.square(y_pred - y_true))) 

MSE= tf.keras.losses.mean_squared_error
MAE = tf.keras.losses.mean_absolute_error
MAPE = tf.keras.losses.mean_absolute_percentage_error

**Optimizers**  

In [ ]:
def get_optimizer():
    optimizer1 = tf.keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999, epsilon=1e-08, amsgrad=True, name="Adam")
    #optimizer1 = tf.keras.optimizers.RMSprop(learning_rate=0.001)
    return optimizer1
    #

**Train options callbacks**  

In [ ]:
# Avalia se está tendo avanção de desempenho no treinamento/validação e para caso não tenha avanço 
EarlyStopping=tf.keras.callbacks.EarlyStopping( monitor="val_loss", patience=20, verbose=1, mode="min", restore_best_weights=True,)

# verifica se está tendo avanço de desempenho durante o treinamento, caso não reduz integralmente o lr
reduceLr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor = 0.01, patience = 5, min_lr=1e-20,verbose=1)


checkpoint_filepath = './models_checkpoint/'
checkpoint_filepath =f'models/{model_name}_{ticker}_{lookback}_ex1'


def checkpoints(model_name):
    checkpoint_filepath =f'models/model_{model_name}_stock_{ticker}_lookback_{lookback}'
    csvLogger = tf.keras.callbacks.CSVLogger(checkpoint_filepath+'_history.csv', separator=',',append=True)
    model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_filepath, verbose=1, save_weights_only=False, monitor='val_loss', mode='min', save_best_only=True)
    print("Training model :", checkpoint_filepath)

    return csvLogger, model_checkpoint_callback

### Models train

In [ ]:
#ategorical_accuracy =tf.keras.metrics.categorical_accuracy
#loss =tf.keras.losses.categorical_crossentropy
loss = weighted_categorical_crossentropy_loss
#loss = tf.keras.losses.MAE

In [ ]:
list_of_models=[model_MLP, model_GRU, model_CNN_LSTM, model_CNN_MultiHead, model_LSTM, model_LSTM_AT]
#list_of_models=[model_LSTM, model_LSTM_AT, model_GRU ]
#list_of_models=[model_LSTM_AT]
#list_of_models=[model_MLP, model_CNN_LSTM, model_LSTM_AT]
#list_of_models=[model_CNN_MultiHead]

for model in list_of_models:  
    optimizer = get_optimizer()
    csvLogger, model_checkpoint_callback  = checkpoints(model._name)
    model.compile(loss=[loss], optimizer=optimizer, metrics=['accuracy',matthews_correlation_coefficient])

    print('------------------------------------------------------------------------------------------------------------------------------------------------------------------------')
    print(f'Initing train fo model: {model._name}')
    #history=model.fit( X_data_gen_train, validation_data=X_data_gen_val, epochs=200, batch_size=32, shuffle=False, validation_split=0.3, callbacks=[EarlyStopping, reduceLr, model_checkpoint_callback,csvLogger])
    #history=model.fit_generator( X_data_gen_train, epochs=200, callbacks=[EarlyStopping, reduceLr, model_checkpoint_callback,csvLogger], validation_data=X_data_gen_val, verbose=0)
    #history=model.fit( X_data_gen_train.features, X_data_gen_train.y_classification, epochs=200, verbose=1)
    history = model.fit(
        X_data_gen_train,
        epochs=200,
        callbacks=[EarlyStopping, reduceLr, model_checkpoint_callback, csvLogger],
        validation_data=X_data_gen_val,
        #shuffle=True,
        verbose=2
    )
    # Assuming history_metric, history_metric_val, history are defined elsewhere

    fig, axes = plt.subplots(2, 1, figsize=(15, 8))  # Create subplots programmatically

    # Plot accuracy, sharing x-axis with NRMSE
    axes[0].plot(history.history['accuracy'])
    axes[0].plot(history.history['val_accuracy'])
    axes[0].set_title(f'{model._name}: Accuracy')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_xlabel('Epoch')  # Shared x-axis label
    axes[0].legend(['train', 'validation'], loc='upper left')

    # Plot loss
    axes[1].plot(history.history['loss'])
    axes[1].plot(history.history['val_loss'])
    axes[1].set_title(f'{model._name}: Loss (categorical_crossentropy)')
    axes[1].set_ylabel('Loss')

    # Adjust spacing and margins (optional)
    plt.subplots_adjust(left=0.1, bottom=0.15, right=0.9, top=0.9, wspace=0.2)

    # Fine-tune spacing (optional)
    plt.tight_layout()

    plt.show()

    #history=model.fit( x=X_train, y=y_train, epochs=200, batch_size=256, shuffle=True, validation_split=0.1, callbacks=[EarlyStopping,model_checkpoint_callback, reduceLr])

### Validation

#### Load the trained models


In [ ]:
trained_best_models={}
for model in list_of_models:
    print(model.name)
    checkpoint_filepath =f'models/model_{model._name}_stock_{ticker}_lookback_{lookback}'
    trained_best_models[f'{model._name}']=tf.keras.models.load_model(
        checkpoint_filepath, 
        custom_objects={'loss': weighted_categorical_crossentropy_loss, 'matthews_correlation_coefficient': matthews_correlation_coefficient})

#### Using the validation Dataset

In [ ]:
Y_val_categorical=np.argmax(X_data_gen_val.y_classification,axis=1)
classes, counts=np.unique(Y_val_categorical,return_counts=True) 

fig, ax = plt.subplots()
plt.bar(classes,counts)
ax.set_ylabel('number of classes')
ax.set_title('Val Dataset classes')
plt.show()

In [ ]:
X_data_gen_val = FeaturesDataGenerator(df[df['Date'].isin(T_val)].iloc[:,1:], lookback = lookback, pred_days = pred_days,  buy_sell_threshold=buy_sell_threshold, shuffle= False, batch_size=batch_size, selected_features = features_indicators, data_augmentation=False)


In [ ]:
# get metrics on validation dataset
idxs=np.where(np.argmax(X_data_gen_val.y_classification, axis=1)> -1 )[0]

x_data = np.zeros_like(X_data_gen_val.features)
for i in range(X_data_gen_val.features.shape[0]):
    x_data[i] = X_data_gen_val.norm_minmax(
        X_data_gen_val.features[i],
        minimum=min_norm,
        maximum=max_norm,
        axis=0
    )

import utils
for model_name in trained_best_models:

    label_pred = trained_best_models[model_name].predict(x_data)
    print(model_name,' classification Accuracy',
                    utils.f1_score(np.argmax(X_data_gen_val.y_classification,axis=1),
                    np.argmax(label_pred,axis=1), average='micro'))
#utils.model_metrics(['subida','Descida'], y_test, y_pred, Get_metrics=False)

#utils.plot_confusion_matrix(['subida','Descida'], y_test, y_pred)

In [ ]:
np.argmax(X_data_gen_val.y_classification[idxs],axis=1).shape

In [ ]:
np.apply_along_axis(lambda row: row.shape, axis=1, arr=X_data_gen_val.features[idxs])

In [ ]:
np.zeros_like

In [ ]:
def apply_NomrMinmax(features, minimum= min_norm, maximum= max_norm, axis=0):
    norm_features=np.zeros_like(features)
    for idx in range(len(features)):
        norm_features[idx]=X_data_gen_val.norm_minmax(features[idx], minimum, maximum, axis)
    return norm_features

In [ ]:
model_name = "CNN_MultiHead"  
print('Model name:',model_name)

x_data= X_data_gen_val.apply_NomrMinmax(X_data_gen_val.features[idxs], minimum= min_norm, maximum= max_norm, axis=0)

label_pred = trained_best_models[model_name].predict(x_data)
print(model_name,' classification Accuracy',
                utils.f1_score(np.argmax(X_data_gen_val.y_classification[idxs],axis=1),
                np.argmax(label_pred,axis=1),average="micro"))

cf_matrix = utils.confusion_matrix(np.argmax(X_data_gen_val.y_classification[idxs],axis=1),np.argmax(label_pred,axis=1))
 
try:
    utils.plot_confusion_matrix(gesture_list= ['Hold','Buy','Sell'], cf_matrix=cf_matrix)
except:
    print(cf_matrix)

### Using test dataset

In [ ]:
Y_train_categorical=np.argmax(X_data_gen_test.y_classification,axis=1)
classes, counts=np.unique(Y_train_categorical,return_counts=True) 

fig, ax = plt.subplots()
plt.bar(classes,counts)
ax.set_ylabel('number of classes')
ax.set_title('Test Dataset classes')
plt.show()

In [ ]:
X_data_gen_test = FeaturesDataGenerator(df[df['Date'].isin(T_test)].iloc[:,1:], lookback = lookback, pred_days = pred_days, shuffle= False, batch_size=1, selected_features = features_indicators, data_augmentation=False)
test_data=[]
targets=[]
for X,y in X_data_gen_test:
    #train_data+=[scaler.transform(np.array(X).reshape(-1,1))]
    test_data+=[np.array(X)]
    targets+=[np.argmax(y)]
targets = np.stack(targets)
test_data = np.vstack(test_data)

In [ ]:
label_pred.shape

In [ ]:
# get metrics for all deep learning models 
import utils
for model_name in trained_best_models:
    label_pred = trained_best_models[model_name].predict(test_data)
    print(f'{model_name} metrics:')
    utils.model_average_std_metrics(targets, np.argmax(label_pred,axis=1), Get_metrics= True,  Verbose=True) ;
    print('------------------------------------------------')

In [ ]:
model_name = "CNN_MultiHead"  
print('Model name:',model_name)

label_pred = trained_best_models[model_name].predict(test_data)
print(model_name,' classification Accuracy',utils.f1_score(targets, np.argmax(label_pred,axis=1),average='macro'))

cf_matrix = utils.confusion_matrix(targets,np.argmax(label_pred,axis=1))
 
try:
    utils.plot_confusion_matrix(gesture_list= ['Hold','Buy','Sell'], cf_matrix=cf_matrix)
except:
    print(cf_matrix)

In [ ]:
model_name = "CNN_MultiHead"    
for model_name in list_of_models:
    model_name = model_name._name
    print('Model name:',model_name)

    x_data= X_data_gen_test.apply_NomrMinmax(X_data_gen_test.features, minimum= min_norm, maximum= max_norm, axis=0)

    label_pred = trained_best_models[model_name].predict(x_data)
    print(model_name,' classification Accuracy',
                    utils.f1_score(np.argmax(X_data_gen_test.y_classification,axis=1),
                    np.argmax(label_pred,axis=1),average="micro"))

    cf_matrix = utils.confusion_matrix(np.argmax(X_data_gen_test.y_classification,axis=1),np.argmax(label_pred,axis=1))
    
    try:
        utils.plot_confusion_matrix(gesture_list= ['Hold','Buy','Sell'], cf_matrix=cf_matrix)
    except:
        print(cf_matrix)

In [ ]:
model_name

#### Using Machine Learning models to Predicti next recomendations

In [ ]:
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
import xgboost as xgb
from sklearn.metrics import accuracy_score
import h5py

import joblib

# Define ML models and their names
ML_models = {
    'SVM': SVC(kernel='rbf',decision_function_shape='ovr'),
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(n_estimators=300),
    'Logistic Regression': LogisticRegression(),
    'K-Nearest Neighbors': KNeighborsClassifier(),
    'XGBoost RF': xgb.XGBRFClassifier(n_estimators=300)
}

x_data_train= np.vstack(np.apply_along_axis(lambda row: X_data_gen_train.norm_minmax(row, minimum= min_norm, maximum= max_norm, axis=-1), axis=1, arr=X_data_gen_train.features).T).T
x_data_val= np.vstack(np.apply_along_axis(lambda row: X_data_gen_val.norm_minmax(row, minimum= min_norm, maximum= max_norm, axis=-1), axis=1, arr=X_data_gen_val.features).T).T
x_data_test= np.vstack(np.apply_along_axis(lambda row: X_data_gen_test.norm_minmax(row, minimum= min_norm, maximum= max_norm, axis=-1), axis=1, arr=X_data_gen_test.features).T).T


# Iterate over ML models
for name, model in ML_models.items():
    # Train the model
    model.fit(x_data_train, np.argmax(X_data_gen_train.y_classification, axis=1) )

    # Make predictions on datasets
    train_pred = model.predict(x_data_train)
    val_pred = model.predict(x_data_val)
    test_pred = model.predict(x_data_test)

    # Calculate accuracy on datasets
    train_accuracy = accuracy_score(np.argmax(X_data_gen_train.y_classification, axis=1), train_pred)
    val_accuracy = accuracy_score(np.argmax(X_data_gen_val.y_classification, axis=1), val_pred)
    #test_accuracy = accuracy_score(np.argmax(X_data_gen_test.comput_outputs(df[df['Date'].isin(T_test)].iloc[:,1].values), axis=1), test_pred)
    test_accuracy = accuracy_score(np.argmax(X_data_gen_test.y_classification, axis=1), test_pred)

    # Print results
    print('-------------------------------------------------------------')
    print(f'{name} - Train Classification Accuracy: {train_accuracy}')
    print(f'{name} - Validation Classification Accuracy: {val_accuracy}')
    print(f'{name} - Test Classification Accuracy: {test_accuracy}')

    # Salve o modelo
    #joblib.dump(model, f'models/model_{name}_stock_{ticker}_lookback_{lookback}.joblib')

In [ ]:
model_name = "Decision Tree"  
print('Model name:',model_name)

label_pred = ML_models[model_name].predict(np.vstack(test_data.T).T)

print(model_name,' classification Accuracy',utils.f1_score(targets, label_pred, average='macro'))

cf_matrix = utils.confusion_matrix(targets,label_pred)
 
try:
    utils.plot_confusion_matrix(gesture_list= ['Hold','Buy','Sell'], cf_matrix=cf_matrix)
except:
    print(cf_matrix)

### Get new values to future prediction 

In [ ]:
dias_antes=1
n_steps=7

print("Temos dados até o dia :",T_test.iloc[-1].date())

print("Estamos Fornecendo dados até o dia :",T_test.iloc[-1].date() -datetime.timedelta(days=dias_antes))

print("O modelo fará previsões até o dia: ",T_test.iloc[-1].date() -datetime.timedelta(days=dias_antes) + datetime.timedelta(days=n_steps))

In [ ]:
data_inference=df[df['Date'].isin(T_test.iloc[-lookback-3:])].iloc[:,1:]
#data_inference=df[df['Date'].isin(T_test.iloc[:])].iloc[:,1:]
data_inference.shape


In [ ]:
data_inference.shape

In [ ]:
dataGen_inference = FeaturesDataGenerator(data_inference, lookback = lookback, pred_days = pred_days, shuffle= False, batch_size=1, selected_features = features_indicators, data_augmentation=False)

In [ ]:
trade=['Hold','Buy','Sell']

In [ ]:
data_inference=df[df['Date'].isin(T_test.iloc[-lookback-pred_days:])].iloc[:,1:]
data_inference=df[df['Date'].isin(T_test.iloc[:-8])].iloc[:,1:]
data_inference['Close'].values.astype(np.float32).shape

In [ ]:
x_data_inference=dataGen_inference.comput_features(data_inference, pred_days=0)
x_data_inference.shape

In [ ]:
x_data_inference=dataGen_inference.comput_features(data_inference, pred_days=0)
x_data= np.apply_along_axis(lambda row: X_data_gen_test.norm_minmax(row, minimum= min_norm, maximum= max_norm, axis=0), axis=1, arr=x_data_inference)

model_name='CNN_MultiHead'
label_pred = trained_best_models[model_name].predict(x_data)
trade[np.argmax(label_pred[-1])]

In [ ]:
model_name = "XGBoost RF"  
print('Model name:',model_name)
label_pred = ML_models[model_name].predict(np.vstack(x_data.T).T)
trade[np.argmax(label_pred[-1])]

In [ ]:
np.unique(label_pred, return_counts=True)

### Inference in real time data 

In [ ]:
from datetime import datetime, timedelta
data_atual = datetime.now()
data_atual_formatada = data_atual.strftime('%Y-%m-%d')
dia_anterior = (data_atual - timedelta(days=11)).strftime('%Y-%m-%d')

interval ='4h'

# Baixar e consolidar dados em um DataFrame
cryptos_df = pd.DataFrame()
for crypto in cryptos:
    data = get_binance_data(crypto, start_time=dia_anterior, interval=interval)
    # Ajusta o nome da coluna removendo "USDT" antes de adicionar ao DF
    #data.columns = [crypto.replace('USDT', '')]
    if cryptos_df.empty:
        cryptos_df = data
    else:
        cryptos_df = pd.concat([cryptos_df, data], axis=1)

if not cryptos_df.empty:
    cryptos_df = cryptos_df.rename(columns={
        'open': 'Open',
        'high': 'High',
        'low': 'Low',
        'close': 'Close',
        'volume': 'Volume',
    })
    # Visualização dos dados
    cryptos_df=cryptos_df.rename_axis('Date')
    print(cryptos_df.shape)
else:
    print("Nenhum dado encontrado para o símbolo e intervalo especificados.")

cryptos_df.reset_index(inplace= True)

In [ ]:
dataGen_inference = FeaturesDataGenerator(data_inference, lookback = lookback, pred_days = pred_days, shuffle= False, batch_size=1, selected_features = features_indicators, data_augmentation=False)
x_data_inference=dataGen_inference.comput_features(cryptos_df, pred_days=0)
x_data_inference.shape

In [ ]:
x_data= np.apply_along_axis(lambda row: X_data_gen_test.norm_minmax(row, minimum= min_norm, maximum= max_norm, axis=0), axis=1, arr=x_data_inference)
model_name='CNN_MultiHead'
label_pred = trained_best_models[model_name].predict(x_data)
trade[np.argmax(label_pred[-1])]

In [ ]:
np.argmax(label_pred[:],axis=1)

In [ ]:
model_name = "XGBoost RF"  
print('Model name:',model_name)
label_pred_ml = ML_models[model_name].predict(np.vstack(x_data.T).T)
trade[label_pred_ml[-1]]

In [ ]:
label_pred_ml

In [ ]:
# Dados de exemplo
try:
    data_idx= -x_data_inference.shape[0]
    input_data = cryptos_df['Close'].iloc[data_idx:].values
    date = cryptos_df['Date'].iloc[data_idx:]
    #labels = labelling_method(close_price_list, 60)

    #labels=[trade[label] for label in label_pred_ml]
    labels=[trade[label] for label in np.argmax(label_pred[:],axis=1)]
    

    labels=labels[data_idx:]
    print(np.unique(labels, return_counts=True))
    portfolio_values = backtest(input_data, labels, initial_capital=100000)

    # Gráfico do desempenho
    import plotly.graph_objects as go

    #fig = go.Figure()

    fig = go.Figure(data=[go.Candlestick(x=cryptos_df['Date'].iloc[data_idx:],
                    open=cryptos_df['Open'].iloc[data_idx:],
                    high=cryptos_df['High'].iloc[data_idx:],
                    low=cryptos_df['Low'].iloc[data_idx:],
                    close=cryptos_df['Close'].iloc[data_idx:],
                    name='Candlestick')])

    # Valor do portfólio
    #fig.add_trace(go.Scatter(
    #    x=X_data_gen_test.InputData['Date'],
    #    y=portfolio_values,
    #    mode='lines',
    #    name='Valor do Portfólio'
    #))

    # Configuração do layout
    fig.update_layout(
        title="Backtest de Estratégia de BUY, SELL, HOLD",
        xaxis_title="Data",
        yaxis_title="Valor",
        showlegend=False,
        width=1700,
        height=600,
        margin=dict(l=50, r=50, t=50, b=50)  # Margem esquerda, direita, superior e inferior
    )


    # Adiciona os marcadores BUY, SELL e HOLD
    for i, label in enumerate(labels):
        if label == "Buy":
            fig.add_trace(go.Scatter(
                x=[date.iloc[i]],
                y=[input_data[i] - 0.01*input_data[i]],
                mode='markers',
                marker=dict(color='green', size=10, symbol='triangle-up'),
                name='BUY'
            ))
        elif label == "Sell":
            fig.add_trace(go.Scatter(
                x=[date.iloc[i]],
                y=[input_data[i] + 0.01*input_data[i]],
                mode='markers',
                marker=dict(color='red', size=10, symbol='triangle-down'),
                name='SELL'
            ))
    fig.update_layout(xaxis_rangeslider_visible=False)
    fig.show()
except Exception as e:\
    print("Erro ao executar o código:", str(e))